# 3. Model training

How a training row is constructed, and why every feature sits where it does.

The whole design follows from one gap: **standing at as-of *T* predicting *T+30*, you do not know *T+29*'s sales.** Step 4's baseline uses `lag_1_units` at the prediction date, which is legitimate for a historical counterfactual and invalid here.

So a training row is `(origin t, horizon step h, target = units at t+h)`, and each feature is placed by asking one question — *is this knowable at t?*

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

from app.services.container import Container
from ml.forecasting.baselines import attach_seasonal_reference
from ml.forecasting.config import load_forecast_config
from ml.forecasting.dataset import (
    HORIZON_STEP,
    ORIGIN_DATE,
    TARGET_DATE,
    TARGET_PREFIX,
    HorizonDataset,
    build_history,
    build_horizon_dataset,
)
from ml.forecasting.sampling import sample_series
from ml.forecasting.split import build_origin_split, worst_case_gap_days
from ml.forecasting.train import build_estimator, train_forecaster

repo = Container().data_repository
config = load_forecast_config().smoke()
print("config fingerprint:", config.fingerprint())

## 1. Origin side, target side

Step 3's **availability classes** already answer the knowability question, so the feature placement is not a judgement call — it is a lookup:

| Class | Tables | Readable for a future date? |
|---|---|---|
| `OBSERVED` | `sales_daily`, `inventory`, `competitor_pricing` | No — clamped to as-of |
| `KNOWN_IN_ADVANCE` | `calendar`, `promotions`, `pricing` | **Yes** |
| `STATIC` | `products`, `stores` | Yes |

In [ ]:
sample = sample_series(repo, n_series=config.sampling.n_series, seed=config.sampling.seed)
history = build_history(repo, config, sample)
view = repo.as_of(pd.to_datetime(history["date"]).dt.date.max())

dataset = build_horizon_dataset(history, view, config, sample)
dataset = HorizonDataset(
    frame=attach_seasonal_reference(dataset.frame, history),
    feature_names=[*dataset.feature_names, "seasonal_reference"],
    excluded=dataset.excluded,
)

target_side = [c for c in dataset.feature_names if c.startswith(TARGET_PREFIX)]
origin_side = [c for c in dataset.feature_names if not c.startswith(TARGET_PREFIX)]

print(dataset.describe())
print()
print(f"origin-side features ({len(origin_side)}): {origin_side[:8]} ...")
print(f"target-side features ({len(target_side)}): {target_side[:8]} ...")
print()
print("rows dropped at each stage:")
for stage, count in dataset.excluded.items():
    print(f"  {stage:28s} {count:>10,}")

The `h_` prefix is not cosmetic. It makes the origin/target split visible in the feature-importance table, which is where a misplacement would otherwise hide.

## 2. One row, inspected by hand

The arithmetic that must hold: `target_date == origin_date + h`, the target is units at the target date, and the lag is measured from the **origin**.

In [ ]:
row = dataset.frame.sample(n=1, random_state=3).iloc[0]
panel = history.copy()
panel["date"] = pd.to_datetime(panel["date"])
lookup = panel.set_index(["product_id", "store_id", "date"])["units"]

key = (row["product_id"], row["store_id"])
print(f"series          : {key[0]} @ {key[1]}")
print(f"origin_date     : {row[ORIGIN_DATE].date()}")
print(f"horizon_step    : {row[HORIZON_STEP]}")
print(f"target_date     : {row[TARGET_DATE].date()}  (origin + h = "
      f"{(row[ORIGIN_DATE] + pd.Timedelta(days=int(row[HORIZON_STEP]))).date()})")
print()
print(f"target (units)  : {row['units']:.0f}")
print(f"panel at target : {lookup.get((*key, row[TARGET_DATE])):.0f}   <- must match")
print()
print(f"lag_7_units     : {row.get('lag_7_units')}")
print(f"panel at o - 7d : {lookup.get((*key, row[ORIGIN_DATE] - pd.Timedelta(days=7)))}   <- must match")
print(f"(NOT the target-7d value: {lookup.get((*key, row[TARGET_DATE] - pd.Timedelta(days=7)))})")

That last contrast is the signature bug of a self-join design. Sourcing origin-side features from the target row produces a dataset that trains fine, scores beautifully, and is measuring the wrong thing.

## 3. Horizon steps are drawn at random

Not from a fixed grid. With a grid, the model's splits on `horizon_step` are piecewise-constant — which shows up as a **staircase in the daily forecast path**, and the path is a deliverable.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(dataset.frame[HORIZON_STEP], bins=90)
ax.set_xlabel("horizon step (days ahead)")
ax.set_ylabel("training rows")
ax.set_title(f"{dataset.frame[HORIZON_STEP].nunique()} distinct horizon steps covered")
plt.tight_layout()
plt.show()

## 4. The split needs an embargo

Step 4's split had no need for one, because its features and target shared a date. Here a training origin sitting just before a fold boundary has its **target inside the evaluation window** — the model is fitted on the outcomes it is about to be scored on.

In [ ]:
split = build_origin_split(dataset.frame, config)
print(split.describe())
print()
print(f"worst-case slack with a {config.validation.embargo_days}d embargo: "
      f"{worst_case_gap_days(split, config.max_horizon)} days")

no_embargo = config.model_copy(
    update={"validation": config.validation.model_copy(update={"embargo_days": 0})}
)
object.__setattr__(no_embargo.validation, "embargo_days", 0)
print(f"worst-case slack with no embargo          : "
      f"{worst_case_gap_days(build_origin_split(dataset.frame, no_embargo), config.max_horizon)} days")
print()
print("Negative means a boundary training origin can reach into an evaluation fold.")

The embargo costs real training data — 90 days of origins at every boundary — and that cost is the point. Note the check measures against the **nearest** evaluation fold, not the test fold: calibration and validation sit in between, so measuring against test alone would look safe even with no embargo at all.

## 5. Train, and see what the model leans on

In [ ]:
model = train_forecaster(dataset, build_estimator("lightgbm", seed=42), config, split)
print(model.summary())

importance = model.estimator.feature_importance()
if importance is not None and not importance.empty:
    top = importance.head(18).iloc[::-1]
    colours = [
        "tab:orange" if f.startswith(TARGET_PREFIX) else "tab:blue"
        for f in top["feature"]
    ]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top["feature"], top["importance"], color=colours)
    ax.set_title("Top features - orange is target-side (h_), blue is origin-side")
    plt.tight_layout()
    plt.show()

A healthy split has **both colours present**. All blue would mean the model ignores the plan it was given; all orange would mean it ignores demand history.

---

## Findings

- Feature placement is decided by Step 3's availability classes, not by judgement.
- The `h_` prefix keeps the origin/target split visible where it matters.
- Horizon steps are random, so the forecast path is smooth rather than stepped.
- The embargo is what makes the evaluation honest at a 90-day horizon.

### Next

`04_backtesting.ipynb` — does the accuracy hold up over time, and what do the intervals actually cover?